In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_PATH = PROJECT_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))


# Milestone 5: Risk and Hedging Across Models

This notebook compares model risk and a simple PV01 hedge across five frameworks:

- Black
- SABR-adjusted Black
- shifted Black
- shifted SABR
- Bachelier

The goal is to study not only price differences, but also whether model choice changes hedging interpretation.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from swaption_pricing.hedging import compare_model_hedging
from swaption_pricing.risk import compare_all_model_risks
from swaption_pricing.sabr import SabrParams
from swaption_pricing.types import CurvePoint, SwaptionSpec


In [ ]:
curve = [
    CurvePoint(maturity=1.0, zero_rate=0.0420),
    CurvePoint(maturity=2.0, zero_rate=0.0415),
    CurvePoint(maturity=3.0, zero_rate=0.0410),
    CurvePoint(maturity=4.0, zero_rate=0.0408),
    CurvePoint(maturity=5.0, zero_rate=0.0405),
    CurvePoint(maturity=6.0, zero_rate=0.0403),
    CurvePoint(maturity=7.0, zero_rate=0.0402),
]

spec = SwaptionSpec(
    notional=10_000_000.0,
    expiry=2.0,
    tenor=5.0,
    strike=0.0400,
    pay_frequency=1,
    option_type='payer',
)
sabr_params = SabrParams(alpha=0.0200, beta=0.50, rho=-0.25, nu=0.40)
black_vol = 0.20
shift = 0.03
normal_vol = 0.01
rate_shift = 0.0025


In [ ]:
risk_comparison = compare_all_model_risks(
    curve=curve,
    spec=spec,
    black_vol=black_vol,
    sabr_params=sabr_params,
    shift=shift,
    normal_vol=normal_vol,
)

hedging_comparison = compare_model_hedging(
    curve=curve,
    spec=spec,
    black_vol=black_vol,
    sabr_params=sabr_params,
    shift=shift,
    normal_vol=normal_vol,
    rate_shift=rate_shift,
)

risk_rows = [
    risk_comparison.black,
    risk_comparison.sabr,
    risk_comparison.shifted_black,
    risk_comparison.shifted_sabr,
    risk_comparison.bachelier,
]
hedge_rows = [
    hedging_comparison.black,
    hedging_comparison.sabr,
    hedging_comparison.shifted_black,
    hedging_comparison.shifted_sabr,
    hedging_comparison.bachelier,
]

risk_df = pd.DataFrame([
    {
        'model': row.model,
        'price': row.price,
        'volatility': row.volatility,
        'pv01': row.risk.pv01,
        'vega': row.risk.vega,
        'theta': row.risk.theta,
    }
    for row in risk_rows
])

hedge_df = pd.DataFrame([
    {
        'model': row.model,
        'hedge_rate': row.hedge_rate,
        'hedge_instrument_pv01': row.hedge_instrument_pv01,
        'hedge_ratio': row.hedge_ratio,
        'unhedged_pnl': row.unhedged_pnl,
        'hedge_pnl': row.hedge_pnl,
        'hedged_pnl': row.hedged_pnl,
    }
    for row in hedge_rows
])

risk_df

In [ ]:
hedge_df

In [ ]:
ax = risk_df.plot(x='model', y=['pv01', 'vega', 'theta'], kind='bar', figsize=(10, 4), title='Model Risk Comparison')
ax.set_ylabel('Sensitivity')
plt.xticks(rotation=20)
plt.show()

In [ ]:
ax = hedge_df.plot(x='model', y=['unhedged_pnl', 'hedged_pnl'], kind='bar', figsize=(10, 4), title='Parallel-Shift PnL Before and After Hedge')
ax.set_ylabel('PnL')
plt.xticks(rotation=20)
plt.show()

## Interpretation

- Price differences are only the first layer; the models can also produce different PV01, vega, and theta values.
- A simple swap-based PV01 hedge usually reduces the parallel-shift PnL, but the residual hedged PnL still depends on the pricing model.
- This means model choice affects not only valuation, but also the practical hedging story seen by a rates desk.